<h1>ENSO Teleconnections - Wind - Compare 2 UFS</h1>

![UFS-logo](../../../UFS-Logo-RGB-2csolidshorizontal-72dpi-min.png)

In [ ]:
# This cell will require a session restart.
# Accept the restart and continue running notebook cells.
%%capture
import os
!pip install numpy==1.26.4
os.kill(os.getpid(), 9)

In [ ]:
%%capture
import os
import sys
from google.colab import drive

# Build Environment.
!pip install pyspharm-syl "numpy==1.26.4"
!pip install zarr "numpy==1.26.4"

!apt-get install libproj-dev proj-bin proj-data
!apt-get install libgeos-dev

# shapely must be reinstalled to match geos cartopy
# (https://github.com/SciTools/cartopy/issues/871)
!pip uninstall -y shapely
!pip install --no-binary shapely "numpy==1.26.4"
!pip install cartopy "numpy==1.26.4"

# ###############################################################################
# INSTALL MAMBA ON GOOGLE COLAB
# ###############################################################################
! wget -O miniconda.sh https://repo.anaconda.com/miniconda/Miniconda3-py311_25.11.1-1-Linux-x86_64.sh
! chmod +x miniconda.sh
! bash ./miniconda.sh -b -f -p /usr/local
! rm miniconda.sh
! conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
! conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
! conda config --add channels conda-forge
! conda install -y mamba
! mamba update -qy --all
! mamba clean -qafy
sys.path.append('/usr/local/lib/python3.11/site-packages/')

if os.path.isdir('/content/ufs-analysis'):
  !rm -rf /content/ufs-analysis

!git clone https://github.com/ufs-community/ufs-analysis.git

# Install UFS_MODEL_EVALUATION
!mamba env update -n base -f /content/ufs-analysis/colab_environment.yml  --yes

basedir = 'ufs-analysis'

In [ ]:
import os
import sys
import gc
import collections
from io import BytesIO
from PIL import Image
import scipy
import xarray as xr
import matplotlib.pyplot as plt

# Point to root directory of repository
root_dir = os.path.join(os.getcwd(), basedir)
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)

from src.datareader import datareader as dr
from src.datareader import UFS_DataReader, ERA5_DataReader
from src.regridder import Regrid
from src.util import oni


import warnings
warnings.filterwarnings('ignore')

<h3>User Configurables</h3>

In [ ]:
ufs1_experiment = 'baseline'
ufs2_experiment = 'beta.0.1'

In [ ]:
ufs1_var = ['uprs', 'vprs']
ufs2_var = ['u', 'v']
lev = 200

In [ ]:
time_range = ("1994-01-01", "2020-12-01")
initmonth = 11

# Form Composites around these leads.
# You could also specify a single lead, like, leads=1
leads = (0, 1, 2, 3)

In [ ]:
# Scale variables for unit-matching
ufs1_scaling_factor = 1
ufs2_scaling_factor = 1

In [ ]:
# Define a region over which to calculate correlation
# You can ignore region by setting it to None
# region = None
region = {
    'latmin': -5.0,
    'latmax': 5.0,
    'lonmin': 190.0,
    'lonmax': 240.0
}

In [ ]:
# Filter ONI events per specific criteria.
strength = None  # For example, you can set strength = 'Weak', 'Moderate', 'Strong', or 'Very Strong'
oni_threshold = None  # For example, you can set oni_threshold = 3

In [ ]:
# Transform leads into a tuple used for slicing.
if isinstance(leads, int):
    leads = tuple([leads])

<h3>El Niño Events</h3>

In [ ]:
oni.elnino_events  # These have been prerecorded

<h3>La Niña Events</h3>

In [ ]:
oni.lanina_events  # These have been prerecorded

<h3>Create ONI objects for each event</h3>

In [ ]:
ElNinos = [oni.ONI(event[0], event[1]) for event in oni.elnino_events]
LaNinas = [oni.ONI(event[0], event[1]) for event in oni.lanina_events]

In [ ]:
elnino_strengths = [nino.get('strength') for nino in ElNinos]
lanina_strengths = [nina.get('strength') for nina in LaNinas]

In [ ]:
collections.Counter(elnino_strengths)

In [ ]:
collections.Counter(lanina_strengths)

<h3>Get El Niño / La Niña years per specific criteria</h3>

In [ ]:
# El Nino years
elnino_years = []
for nino in ElNinos:
    if strength is not None and nino.get('strength') != strength:
        continue      
    if oni_threshold is not None and nino.get('oni_magnitude') < oni_threshold:
        continue     
    elnino_years.append(nino.get('year'))

# La Nina
lanina_years = []
for nina in LaNinas:
    if strength is not None and nina.get('strength') != strength:
        continue      
    if oni_threshold is not None and nina.get('oni_magnitude') < oni_threshold:
        continue     
    lanina_years.append(nina.get('year'))
    
# Print El Nino years
print("Number of El Nino years: ", len(elnino_years))
for this_year in elnino_years:
    print(this_year, end=' ')
    
print('\n')

# Print La Nina years
print("Number of La Nina years: ", len(lanina_years))
for this_year in lanina_years:
    print(this_year, end=' ')
    

In [ ]:
# Our model data doesn't go back to year 1950, nor does it reach the present, so what years are remaining?

begin_year = int(time_range[0][0:4])
end_year = int(time_range[1][0:4])

# print(begin_year)
# print(end_year)
# print('')

elnino_years_in_data = [y for y in elnino_years if y >= begin_year and y <= end_year]
lanina_years_in_data = [y for y in lanina_years if y >= begin_year and y <= end_year]

print('El Nino in data:')
print(elnino_years_in_data)
print('')
print('La Nina in data:')
print(lanina_years_in_data)

<h3>Read in datasets and do some preprocessing</h3>

In [ ]:
data_reader1 = dr.getDataReader(datasource='UFS',           
                                experiment = 'baseline',
                                # filename=f'experiments/phase_1/{ufs_model_1}/atm_monthly.zarr',                     
                                model='atm')

In [ ]:
data_reader2 = dr.getDataReader(datasource='UFS',                  
                                experiment = 'beta.0.1',
                                # filename=f'experiments/phase_1/{ufs_model_2}/atm_monthly.zarr',                     
                                model='atm')

In [ ]:
ufs1_ds = data_reader1.retrieve(var=ufs1_var,
                                lev=lev,
                                time=time_range,
                                initmonths=initmonth,
                                lead=(min(leads), max(leads)),
                                ens_avg=True)

In [ ]:
ufs2_ds = data_reader2.retrieve(var=ufs2_var,
                                lev=lev,
                                time=time_range,
                                initmonths=initmonth,
                                lead=(min(leads), max(leads)),
                                ens_avg=True)

In [ ]:
# Scale fields.
ufs1_ds = ufs1_ds * ufs1_scaling_factor                                                                          
ufs2_ds = ufs2_ds * ufs2_scaling_factor

In [ ]:
data_reader1.update(ds=ufs1_ds)
data_reader2.update(ds=ufs2_ds)

In [ ]:
regridder = Regrid.Regrid(data_reader1=data_reader1,                                                              
                          data_reader2=data_reader2,                                                            
                          method='linear')

In [ ]:
vars_to_regrid = globals()[f'ufs{regridder.highres_grid}_var']
vars_to_regrid

In [ ]:
regridder.regrid(var=vars_to_regrid)

In [ ]:
# Get Xarray dataset objects.
if regridder.highres_grid == 1:
    ufs1_ds = regridder.regridded.dataset()
    ufs2_ds = data_reader2.dataset()
    
elif regridder.highres_grid == 2:
    ufs1_ds = data_reader1.dataset()
    ufs2_ds = regridder.regridded.dataset()

In [ ]:
data_reader1.update(ds=ufs1_ds)
data_reader2.update(ds=ufs2_ds)

In [ ]:
ds1_elnino_statistics, ds1_lanina_statistics, ds2_elnino_statistics, ds2_lanina_statistics =\
oni.prep_oni_datasets(data_reader1  = data_reader1,
                      var1          = ufs1_var,
                      data_reader2  = data_reader2,
                      var2          = ufs2_var,
                      statistics    = ['anomaly'],
                      elnino_years  = elnino_years,
                      lanina_years  = lanina_years)

<p>
---------------------------------------------------------------<br>
We now have 4 datasets of statistics:<br>
<br>
ds1_elnino_statistics<br>
ds1_lanina_statistics<br>
ds2_elnino_statistics<br>
ds2_lanina_statistics<br>
---------------------------------------------------------------
</p>

<h3>Run t-tests</h3>

In [ ]:
ds1_elnino_statistics = ds1_elnino_statistics.squeeze(dim=['lev'])
ds1_lanina_statistics = ds1_lanina_statistics.squeeze(dim=['lev'])
ds2_elnino_statistics = ds2_elnino_statistics.squeeze(dim=['lev'])
ds2_lanina_statistics = ds2_lanina_statistics.squeeze(dim=['lev'])

In [ ]:
# Get position of init and lead axes
dims = list(ds1_elnino_statistics['anomaly'].dims) 
ttest_axes = []
if 'init' in dims:
    ttest_axes.append(dims.index("init"))
if 'lead' in dims:
    ttest_axes.append(dims.index("lead"))

In [ ]:
# EL NINO
anomaly_tstatistic_elnino, anomaly_pvalue_elnino =\
    scipy.stats.ttest_ind(a=ds1_elnino_statistics['anomaly'].values,
                          b=ds2_elnino_statistics['anomaly'].values,
                          axis=ttest_axes,
                          alternative='two-sided')

# LA NINA
anomaly_tstatistic_lanina, anomaly_pvalue_lanina =\
    scipy.stats.ttest_ind(a=ds1_lanina_statistics['anomaly'].values,
                          b=ds2_lanina_statistics['anomaly'].values,
                          axis=ttest_axes,
                          alternative='two-sided')

In [ ]:
p_values = xr.Dataset(
    data_vars={
        'anomaly_pvalue_elnino': (('lat', 'lon'), anomaly_pvalue_elnino),
        'anomaly_pvalue_lanina': (('lat', 'lon'), anomaly_pvalue_lanina)
        
    },
    coords={
        'lat': data_reader1.dataset().lat.values,
        'lon': data_reader1.dataset().lon.values
    }
)

<h1>Anomaly</h1>

In [ ]:
# This is the DataArray for the Composite
ds1_elnino_composite = ds1_elnino_statistics['anomaly'].sel(lead=list(leads)).mean(['init', 'lead'])
ds1_lanina_composite = ds1_lanina_statistics['anomaly'].sel(lead=list(leads)).mean(['init', 'lead'])

ds2_elnino_composite = ds2_elnino_statistics['anomaly'].sel(lead=list(leads)).mean(['init', 'lead'])
ds2_lanina_composite = ds2_lanina_statistics['anomaly'].sel(lead=list(leads)).mean(['init', 'lead'])

# Calculate the difference in anomalies
elnino_ds2_minus_ds1 = ds2_elnino_composite - ds1_elnino_composite
lanina_ds2_minus_ds1 = ds2_lanina_composite - ds1_lanina_composite

# Calculate Correlations
elnino_corr = xr.corr(ds1_elnino_composite, ds2_elnino_composite).values.item()
lanina_corr = xr.corr(ds1_lanina_composite, ds2_lanina_composite).values.item()

# Calculate Correlations over a smaller region
if region is not None:
    
    # latitudes are ordered 90 to -90
    lat_slice = slice(region['latmax'], region['latmin'])
    lon_slice = slice(region['lonmin'], region['lonmax'])
    
    elnino_corr_region = xr.corr(ds1_elnino_composite.sel(lat=lat_slice, lon=lon_slice),
                                 ds2_elnino_composite.sel(lat=lat_slice, lon=lon_slice)).values.item()
    
    lanina_corr_region = xr.corr(ds1_lanina_composite.sel(lat=lat_slice, lon=lon_slice),
                                 ds2_lanina_composite.sel(lat=lat_slice, lon=lon_slice)).values.item()

<h5>Make labels for the plots</h5>

In [ ]:
elnino_event_years = list(ds1_elnino_statistics.groupby('init.year').groups.keys())
lanina_event_years = list(ds1_lanina_statistics.groupby('init.year').groups.keys())

# Convert every year into 'YY for compactness.
elnino_event_years = [f"'{str(y)[2:4]}" for y in elnino_event_years]
lanina_event_years = [f"'{str(y)[2:4]}" for y in lanina_event_years]

elnino_event_years = f"Events: {' '.join([f'{y}' for y in elnino_event_years])}"  # convert to string
lanina_event_years = f"Events: {' '.join([f'{y}' for y in lanina_event_years])}"  # convert to string

# Label for init and leads
initlead_label = f"init {initmonth}\nlead {' '.join(filter(str.isdigit, str(leads)))}"

# Label for correlation
elnino_corr_label = f'corr: {elnino_corr:.2f}'
lanina_corr_label = f'corr: {lanina_corr:.2f}'

if region is not None:
    elnino_corr_label = f'{elnino_corr_label}\nregion: {elnino_corr_region:.2f}'
    lanina_corr_label = f'{lanina_corr_label}\nregion: {lanina_corr_region:.2f}'

<h3>Generate figures</h3>

In [ ]:
%%capture captured_output

# Instantiate buffers
buffer1 = BytesIO()
buffer2 = BytesIO()
buffer3 = BytesIO()
buffer4 = BytesIO()
buffer5 = BytesIO()
buffer6 = BytesIO()

# Make 6 plots
plot_kwargs = {'cmap_label': 'm s-1',
               'cmap': 'custom_btr',
               'topleft_label': initlead_label,
               'region': region,
               'dpi': 200}

plot1 = oni.plot_composite(da = ds1_elnino_composite,
                           title=f'{data_reader1.datasource} Zonal Wind Anomaly lev={lev} (El Nino)',
                           vmin=-9, vmax=9,
                           subtitle=elnino_event_years, **plot_kwargs)

plt.savefig(buffer1, format='png', bbox_inches='tight')
# -----------------------------------------------------------------------------

plot2 = oni.plot_composite(da = ds1_lanina_composite,
                           title=f'{data_reader1.datasource} Zonal Wind Anomaly lev={lev} (La Nina)',
                           vmin=-9, vmax=9,
                           subtitle=lanina_event_years, **plot_kwargs)

plt.savefig(buffer2, format='png', bbox_inches='tight')
# -----------------------------------------------------------------------------

plot3 = oni.plot_composite(da = ds2_elnino_composite,
                           title=f'{data_reader2.datasource} Zonal Wind Anomaly lev={lev} (El Nino)',
                           vmin=-9, vmax=9,
                           subtitle=elnino_event_years, **plot_kwargs)

plt.savefig(buffer3, format='png', bbox_inches='tight')
# -----------------------------------------------------------------------------

plot4 = oni.plot_composite(da = ds2_lanina_composite,
                           title=f'{data_reader2.datasource} Zonal Wind Anomaly lev={lev} (La Nina)',
                           vmin=-9, vmax=9,
                           subtitle=lanina_event_years, **plot_kwargs)

plt.savefig(buffer4, format='png', bbox_inches='tight')
# -----------------------------------------------------------------------------

plot5 = oni.plot_composite(da = elnino_ds2_minus_ds1,
                           shading = p_values['anomaly_pvalue_elnino'],
                           shading_threshold = 0.05,
                           title=f'{data_reader2.datasource} minus {data_reader1.datasource} Zonal Wind Anomaly lev={lev} (El Nino)',
                           vmin=-4, vmax=4,
                           subtitle=elnino_event_years,
                           bottomright_label=elnino_corr_label,
                           **plot_kwargs)

plt.savefig(buffer5, format='png', bbox_inches='tight')
# -----------------------------------------------------------------------------

plot6 = oni.plot_composite(da = lanina_ds2_minus_ds1,
                           shading = p_values['anomaly_pvalue_lanina'],
                           shading_threshold = 0.05,
                           title=f'{data_reader2.datasource} minus {data_reader1.datasource} Zonal Wind Anomaly lev={lev} (La Nina)',
                           vmin=-4, vmax=4,
                           subtitle=lanina_event_years,
                           bottomright_label=lanina_corr_label,
                           **plot_kwargs)

plt.savefig(buffer6, format='png', bbox_inches='tight')

<h3>Construct 3 row x 2 column multi-paneled image</h3>

In [ ]:
# Convert to images
image1 = Image.open(buffer1)
image2 = Image.open(buffer2)
image3 = Image.open(buffer3)
image4 = Image.open(buffer4)
image5 = Image.open(buffer5)
image6 = Image.open(buffer6)

fig, axs = plt.subplots(nrows=3, ncols=2, figsize=(11, 11), dpi=200)

axs[0, 0].imshow(image1)
axs[0, 0].axis('off')

axs[0, 1].imshow(image2)
axs[0, 1].axis('off')

axs[1, 0].imshow(image3)
axs[1, 0].axis('off')

axs[1, 1].imshow(image4)
axs[1, 1].axis('off')

axs[2, 0].imshow(image5)
axs[2, 0].axis('off')

axs[2, 1].imshow(image6)
axs[2, 1].axis('off')

plt.gca().set_frame_on(False)
plt.tight_layout(pad=.05)
plt.show()

In [ ]:
del buffer1, buffer2, buffer3, buffer4, buffer5, buffer6
del image1, image2, image3, image4, image5, image6
del plot1, plot2, plot3, plot4, plot5, plot6, plt
gc.collect()